In [4]:
import pandas as pd

try:
    movies = pd.read_csv('movies.csv', dtype={'movieId': 'int64'})
    ratings = pd.read_csv('ratings.csv', dtype={'movieId': 'int64', 'rating': 'float64'})
    display(movies.head())
    display(ratings.head())
except FileNotFoundError:
    print("Error: One or both of the CSV files were not found.")
except pd.errors.ParserError:
    print("Error: There was an issue parsing the CSV file(s).  Check the file format.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


# Create a user-item matrix
user_item_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
user_item_matrix = user_item_matrix.fillna(0) # Fill missing values with 0

# Calculate user similarity
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

def find_similar_users(target_user_id, user_similarity_df, k):
    # Get similarity scores for the target user
    if target_user_id not in user_similarity_df.index:
        print(f"Error: User ID {target_user_id} not found in the dataset.")
        return pd.Series([]) # Return an empty Series

    target_user_similarities = user_similarity_df.loc[target_user_id].drop(target_user_id, errors='ignore')

    # Sort by similarity and get top k
    similar_users = target_user_similarities.sort_values(ascending=False).head(k)

    return similar_users

def recommend_movies(target_user_id, user_item_matrix, similar_users, num_recommendations, movies_df):
    # Get movies rated by the target user
    if target_user_id not in user_item_matrix.index:
        print(f"Error: User ID {target_user_id} not found in the dataset.")
        return [] # Return an empty list

    target_user_rated_movies = user_item_matrix.loc[target_user_id]
    target_user_rated_movies = target_user_rated_movies[target_user_rated_movies > 0].index.tolist()

    # Aggregate ratings from similar users
    recommendations = {}
    for user_id, similarity_score in similar_users.items():
        if user_id in user_item_matrix.index: # Check if similar user exists in the matrix
            user_ratings = user_item_matrix.loc[user_id]
            for movie_id, rating in user_ratings.items():
                if movie_id not in target_user_rated_movies and rating > 0:
                    if movie_id not in recommendations:
                        recommendations[movie_id] = 0
                    recommendations[movie_id] += rating * similarity_score

    # Sort recommendations by score and get top N
    recommended_movies_with_scores = sorted(recommendations.items(), key=lambda item: item[1], reverse=True)[:num_recommendations]

    # Get movie titles for the recommended movie IDs
    recommended_movie_titles = []
    for movie_id, score in recommended_movies_with_scores:
        movie_title = movies_df[movies_df['movieId'] == movie_id]['title'].values
        if len(movie_title) > 0:
            recommended_movie_titles.append((movie_id, movie_title[0]))

    return recommended_movie_titles


# Set the parameters for the specific user
target_user_id = 1
num_recommendations = 5
k_similar_users = 100

# Find similar users
similar_users = find_similar_users(target_user_id, user_similarity_df, k_similar_users)

# Generate recommendations
recommended_movies_info = recommend_movies(target_user_id, user_item_matrix, similar_users, num_recommendations, movies)

# Print the recommendations in tabular format
print(f"Top {num_recommendations} movie recommendations for user {target_user_id} based on {k_similar_users} similar users:")
if recommended_movies_info:
    # Create a list of dictionaries for the DataFrame
    recommendations_data = []
    for i, (movie_id, movie_title) in enumerate(recommended_movies_info):
        recommendations_data.append({'Sno': i + 1, 'Movie Title': movie_title})

    # Create a Pandas DataFrame from the list of dictionaries
    recommendations_df = pd.DataFrame(recommendations_data)

    # Display the DataFrame in a tabular format
    display(recommendations_df)
else:
    print("No recommendations found.")

Top 5 movie recommendations for user 1 based on 100 similar users:


,Sno,Movie Title
0,1,Indiana Jones and the Last Crusade (1989)
1,2,Toy Story (1995)
2,3,Memento (2000)
3,4,Die Hard (1988)
4,5,Aliens (1986)
